# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subata24/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{}');".format(os.environ["HF_TOKEN"]))

label_df = con.sql("""
WITH feb AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    mar.client_hash_id,
    mar.content_hash_id,
    feb.clicks_feb,
    mar.clicks_mar,
    CASE WHEN mar.clicks_mar < feb.clicks_feb THEN 1 ELSE 0 END AS declined
FROM mar
JOIN feb ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
""").df()

print(label_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 5)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#**My Rule**

Pages that have gone stale (little to no update since creation, past the first ~90 days) and are underperforming their search ranking position on CTR should be prioritized for review. Both signals point toward the same outcome — a page that search users are seeing but not clicking, sitting untouched — which is exactly the kind of page a content refresh is meant to fix. The baseline score combines staleness and CTR-underperformance to rank pages most likely to benefit from a refresh; CTR-outperforming pages are excluded from this reasoning, since that bucket showed a decline spike that looks like low-impression noise rather than a real effect.

##**Signal 1 — Staleness (content age), behind refresh flags**

Hypothesis: older content is more likely to be declining. Bucketed content age (days since content_created_date) against the decline label.

| Bucket | n | decline_rate |
|---|---|---|
| <90 days | 39,681 | 0.135 |
| 90-365 days | 83,358 | 0.202 |
| 1-2 years | 11,199 | 0.199 |

No content in this dataset is older than 2 years. Decline rate rises from 13.5% to ~20% once content passes 90 days old, then plateaus rather than climbing further with age.

**Verdict: CONFIRMED** — directionally supports staleness as a real signal (new content is protected), though the effect is a step-change around 90 days rather than a continuous aging trend.

In [10]:
con.register("label_df_tbl", label_df)

staleness_df = con.sql("""
SELECT
    l.declined,
    DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
    CASE
        WHEN DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') < 90 THEN '1: <90 days'
        WHEN DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') < 365 THEN '2: 90-365 days'
        WHEN DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') < 730 THEN '3: 1-2 years'
        ELSE '4: 2+ years'
    END AS age_bucket
FROM label_df_tbl l
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON l.client_hash_id = dc.client_hash_id AND l.content_hash_id = dc.content_hash_id
WHERE dc.content_created_date IS NOT NULL
""").df()

bucket_table = staleness_df.groupby("age_bucket").agg(
    n=("declined", "size"),
    decline_rate=("declined", "mean")
).round(3)

print(bucket_table)

                    n  decline_rate
age_bucket                         
1: <90 days     39681         0.135
2: 90-365 days  83358         0.202
3: 1-2 years    11199         0.199


##**Signal 2 — CTR-vs-position, behind CTR-fix logic**

Hypothesis: pages whose CTR underperforms what their ranking position predicts are more likely to be declining. Compared actual Feb CTR against a rough expected-CTR curve by position bucket, then checked decline rate.

| Bucket | n | decline_rate |
|---|---|---|
| underperforming | 82,812 | 0.196 |
| as expected | 49,841 | 0.141 |
| outperforming | 1,585 | 0.789 |

The first two buckets support the hypothesis directionally — underperforming CTR correlates with a higher decline rate than as-expected CTR. The "outperforming" bucket's much higher decline rate (78.9%) on a far smaller n is likely a low-impression noise artifact (a handful of impressions can produce an inflated CTR by chance), not a real effect — this bucket would need a minimum-impressions filter to trust.

**Verdict: MIXED** — directionally useful for the underperforming case, but not reliable across all three buckets without further filtering.

In [11]:
ctr_df = con.sql("""
SELECT
    l.declined,
    fp.gsc_avg_position,
    CASE WHEN fp.gsc_impressions > 0 THEN fp.gsc_clicks * 1.0 / fp.gsc_impressions ELSE NULL END AS ctr
FROM label_df_tbl l
JOIN (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS gsc_avg_position,
           SUM(gsc_clicks) AS gsc_clicks,
           SUM(gsc_impressions) AS gsc_impressions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
) fp ON l.client_hash_id = fp.client_hash_id AND l.content_hash_id = fp.content_hash_id
WHERE fp.gsc_impressions > 0
""").df()

# Expected CTR given position (rough industry curve — position 1 gets highest CTR, falls off after)
import numpy as np
ctr_df["expected_ctr"] = np.where(ctr_df["gsc_avg_position"] <= 3, 0.25,
                          np.where(ctr_df["gsc_avg_position"] <= 10, 0.05, 0.01))
ctr_df["ctr_gap"] = ctr_df["ctr"] - ctr_df["expected_ctr"]

ctr_df["ctr_bucket"] = np.where(ctr_df["ctr_gap"] < -0.02, "1: underperforming",
                        np.where(ctr_df["ctr_gap"] < 0.02, "2: as expected", "3: outperforming"))

bucket_table_ctr = ctr_df.groupby("ctr_bucket").agg(
    n=("declined", "size"),
    decline_rate=("declined", "mean")
).round(3)

print(bucket_table_ctr)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                        n  decline_rate
ctr_bucket                             
1: underperforming  82812         0.196
2: as expected      49841         0.141
3: outperforming     1585         0.789


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

##**The rule, encoded:**

- **stale_flag** = 1 if content_age_days >= 90 (matches the step-change found in Signal 1)
- **low_ctr_flag** = 1 if CTR is more than 2 points below the expected CTR for its position bucket, AND impressions >= 10 (the impressions floor was added specifically to exclude the noisy "outperforming" bucket found in Signal 2)
- **score** = stale_flag + low_ctr_flag (0, 1, or 2)
- **reason_code** (one per row): STALE_LOW_CTR, STALE_ONLY, LOW_CTR_ONLY, or NO_FLAG
- **action**: score 2 → REFRESH_PRIORITY, score 1 → MONITOR, score 0 → NO_ACTION

Result: 47,672 REFRESH_PRIORITY, 86,513 MONITOR, 19,374 NO_ACTION.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the combined feature set for scoring
staleness_agg = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    DATE_DIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
WHERE content_created_date IS NOT NULL
""").df()

feb_ctr_agg = con.sql("""
SELECT client_hash_id, content_hash_id,
       AVG(gsc_avg_position) AS gsc_avg_position,
       SUM(gsc_clicks) AS gsc_clicks,
       SUM(gsc_impressions) AS gsc_impressions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

import numpy as np

queue = feb_ctr_agg.merge(staleness_agg, on=["client_hash_id", "content_hash_id"], how="inner")

# Signal 1: staleness flag — matches the step-change found at 90 days
queue["stale_flag"] = (queue["content_age_days"] >= 90).astype(int)

# Signal 2: CTR-underperformance flag — with a minimum-impressions floor to exclude noisy pages
MIN_IMPRESSIONS = 10
queue["ctr"] = np.where(queue["gsc_impressions"] > 0, queue["gsc_clicks"] / queue["gsc_impressions"], np.nan)
queue["expected_ctr"] = np.where(queue["gsc_avg_position"] <= 3, 0.25,
                          np.where(queue["gsc_avg_position"] <= 10, 0.05, 0.01))
queue["ctr_gap"] = queue["ctr"] - queue["expected_ctr"]

queue["low_ctr_flag"] = np.where(
    queue["gsc_impressions"] < MIN_IMPRESSIONS, 0,   # not enough data — don't flag
    (queue["ctr_gap"] < -0.02).astype(int)
)

# Score = sum of the two flags (0, 1, or 2)
queue["score"] = queue["stale_flag"] + queue["low_ctr_flag"]

# Reason code — ONE per row
def reason_code(row):
    if row["stale_flag"] == 1 and row["low_ctr_flag"] == 1:
        return "STALE_LOW_CTR"
    elif row["stale_flag"] == 1:
        return "STALE_ONLY"
    elif row["low_ctr_flag"] == 1:
        return "LOW_CTR_ONLY"
    else:
        return "NO_FLAG"

queue["reason_code"] = queue.apply(reason_code, axis=1)

# Action label
def action_label(score):
    if score == 2:
        return "REFRESH_PRIORITY"
    elif score == 1:
        return "MONITOR"
    else:
        return "NO_ACTION"

queue["action"] = queue["score"].apply(action_label)

# Rank: highest score first, then largest CTR gap (more negative = worse) as tiebreaker
queue_ranked = queue.sort_values(["score", "ctr_gap"], ascending=[False, True]).reset_index(drop=True)

print(queue_ranked.shape)
print(queue_ranked["action"].value_counts())
print(queue_ranked[["client_hash_id", "content_hash_id", "score", "reason_code", "action"]].head(10))

(153559, 14)
action
MONITOR             86513
REFRESH_PRIORITY    47672
NO_ACTION           19374
Name: count, dtype: int64
            client_hash_id           content_hash_id  score    reason_code  \
0  client_e547b89c05043229  content_a7b84aac7fa16f7f      2  STALE_LOW_CTR   
1  client_e547b89c05043229  content_0537cf3c2786de4f      2  STALE_LOW_CTR   
2  client_e547b89c05043229  content_6ad6b98671978801      2  STALE_LOW_CTR   
3  client_e547b89c05043229  content_208771d2c5ba3253      2  STALE_LOW_CTR   
4  client_e547b89c05043229  content_9b5680d845946b6b      2  STALE_LOW_CTR   
5  client_e547b89c05043229  content_99dbde256f521c1e      2  STALE_LOW_CTR   
6  client_e547b89c05043229  content_0c7085879651b36f      2  STALE_LOW_CTR   
7  client_e547b89c05043229  content_c99851b683c7780f      2  STALE_LOW_CTR   
8  client_e547b89c05043229  content_a1940ef31c0ce828      2  STALE_LOW_CTR   
9  client_e547b89c05043229  content_69aaa690a1d8de91      2  STALE_LOW_CTR   

             acti

In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = ["client_hash_id", "content_hash_id", "content_age_days", "gsc_avg_position",
               "ctr", "expected_ctr", "ctr_gap", "stale_flag", "low_ctr_flag",
               "score", "reason_code", "action"]

queue_ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written:", os.path.abspath("work/outputs/baseline_action_score.csv"))
print(queue_ranked.shape[0], "rows written")

Written: /content/work/outputs/baseline_action_score.csv
153559 rows written




## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

###All 20 rows: action = REFRESH_PRIORITY, reason_code = STALE_LOW_CTR, ctr = 0.0 (zero clicks despite passing the 10-impression floor — real visibility, no engagement).

1. content_1a61e3b939c4207d — 200 days old, position 2.4, 0 CTR. Confidence: high — strong position with zero clicks over real impressions is a clear snippet/title problem. Wrong if: this page is a policy/legal page not meant to be clicked (e.g. terms of service).
2. content_49f414e0566e81b5 — 200 days old, position 1.8 (near top of page 1), 0 CTR. Confidence: high — top-3 position with zero clicks is the strongest possible CTR-fix candidate. Wrong if: impressions are bot/crawler traffic, not real users.
3. content_72dc669632c1805e — 200 days old, position 2.4, 0 CTR. Confidence: high. Wrong if: the page was recently deleted/redirected and GSC is reporting stale crawl data.
4. content_b142c6f76faa348f — 130 days old, position 2.6, 0 CTR. Confidence: high. Wrong if: this keyword's actual search intent doesn't match this page (mismatched query-to-page mapping, not a staleness problem).
5. content_71155935954842e2 — 130 days old, position 2.6, 0 CTR. Confidence: high. Wrong if: same as above — intent mismatch rather than content decay.
6. content_40b1af398ec0569a — 130 days old, position 1.2 (page 1 top), 0 CTR. Confidence: high — near-#1 position with zero clicks is unusual and worth manual review first. Wrong if: this is a duplicate/canonical page GSC is misattributing impressions to.
7. content_bff860bf1cd9e2b8 — 130 days old, position 2.6, 0 CTR. Confidence: high. Wrong if: seasonal keyword with a temporary demand dip unrelated to the page itself.
8. content_a46de8cf17851cca — 130 days old, position 1.1 (page 1 top), 0 CTR. Confidence: high — very strong position, zero clicks stands out. Wrong if: the SERP snippet already fully answers the query, so users don't need to click through (a "good" zero-click outcome).
9. content_d9d32bb51a7ca8b9 — 130 days old, position 0.7 (#1 spot), 0 CTR. Confidence: highest in this batch — #1 position with zero clicks is the clearest signal in the queue. Wrong if: this is a featured-snippet-only ranking where GSC counts impressions without a real click opportunity.
10. content_74dc0dd15f630d1d — 130 days old, position 0.9 (#1 spot), 0 CTR. Confidence: highest — same pattern as #9. Wrong if: same featured-snippet caveat.
11. content_64b82d96fc748c44 — 130 days old, position 2.9, 0 CTR. Confidence: high. Wrong if: intent mismatch.
12. content_659c8af08f43aaed — 130 days old, position 0.8 (#1 spot), 0 CTR. Confidence: highest. Wrong if: featured-snippet caveat.
13. content_a1c645ed8f122af7 — 130 days old, position 2.9, 0 CTR. Confidence: high. Wrong if: intent mismatch.
14. content_50131f4f4c2bc56d — 128 days old, position 2.9, 0 CTR. Confidence: high. Wrong if: intent mismatch.
15. content_66079979f34944a7 — 128 days old, position 2.6, 0 CTR. Confidence: high. Wrong if: intent mismatch.
16. content_6097bc9806a3272b — 128 days old, position 1.5, 0 CTR. Confidence: high. Wrong if: page recently redirected.
17. content_1188f21a68199f41 — 128 days old, position 1.2, 0 CTR. Confidence: high. Wrong if: bot traffic inflating impressions.
18. content_5fad371622411cc7 — 128 days old, position 3.0 (edge of top-3), 0 CTR. Confidence: medium-high — right at the position-bucket boundary, expected-CTR estimate is less precise here. Wrong if: position 3.0 is borderline between the 0.25 and 0.05 expected-CTR tiers used in this rule, so the "underperforming" call is more sensitive to bucket edges here.
19. content_9390673ca64811e7 — 128 days old, position 1.9, 0 CTR. Confidence: high. Wrong if: intent mismatch.
20. content_c10d96bcd044224e — 284 days old (different client), position 2.3, 0 CTR. Confidence: high — oldest page in this batch, and the only client outside the top-19 cluster, confirming the clustering issue named in Section 4. Wrong if: intent mismatch or bot traffic.

In [14]:
top20 = queue_ranked.head(20)[["client_hash_id", "content_hash_id", "content_age_days",
                                 "gsc_avg_position", "ctr", "expected_ctr", "score",
                                 "reason_code", "action"]]
print(top20.to_string())

             client_hash_id           content_hash_id  content_age_days  gsc_avg_position  ctr  expected_ctr  score    reason_code            action
0   client_e547b89c05043229  content_a7b84aac7fa16f7f               345          0.215986  0.0          0.25      2  STALE_LOW_CTR  REFRESH_PRIORITY
1   client_e547b89c05043229  content_0537cf3c2786de4f               445          2.077349  0.0          0.25      2  STALE_LOW_CTR  REFRESH_PRIORITY
2   client_e547b89c05043229  content_6ad6b98671978801               345          2.420193  0.0          0.25      2  STALE_LOW_CTR  REFRESH_PRIORITY
3   client_e547b89c05043229  content_208771d2c5ba3253               387          2.517373  0.0          0.25      2  STALE_LOW_CTR  REFRESH_PRIORITY
4   client_e547b89c05043229  content_9b5680d845946b6b               387          0.747619  0.0          0.25      2  STALE_LOW_CTR  REFRESH_PRIORITY
5   client_e547b89c05043229  content_99dbde256f521c1e               387          0.412099  0.0          0.



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick — the tie-break clustering problem**

19 of the top 20 rows belong to a single client (client_e547b89c05043229). This isn't a data error — the underlying scores are individually defensible (each page really is stale, really has 0 CTR on 10+ impressions) — but the ranking's tiebreaker (ctr_gap ascending) doesn't discriminate once ctr = 0.0 for every tied row, so ties fall back to row order, which happens to cluster by client. A real editor queue dominated by one client isn't fair or useful across FlyRank's book of business. Fix for next iteration: break ties using a signal that varies even at ctr=0 — e.g. impression volume (prioritize higher-visibility zero-click pages first) or add a per-client cap on how many REFRESH_PRIORITY picks can come from one client in the top N.

**Weak pick — the featured-snippet blind spot**

Several top-20 rows sit at position 0.7-1.5 (effectively #1) with 0 CTR (rows 6, 8, 9, 10, 12, 16, 17). A #1 ranking with zero clicks is unusual enough to be the strongest possible signal — but it's also consistent with a page that already fully answers the query in a featured snippet, meaning users get their answer without clicking through. That's a "good" zero-click outcome, not a decaying page, and this rule can't currently tell the difference. Confirming this would require checking SERP feature data this rule doesn't have access to.

**Weak pick — position-bucket boundary sensitivity**

Row 18 (content_5fad371622411cc7) sits at position 3.0, right at the edge between the "position <=3" (expected CTR 0.25) and "position <=10" (expected CTR 0.05) buckets used in the rule. A page just barely inside or outside this boundary gets a very different expected-CTR baseline, which can flip the low_ctr_flag on a marginal difference in ranking rather than a real change in performance. A smoother expected-CTR curve (rather than hard buckets) would reduce this sensitivity.

**Leakage check**

- stale_flag uses content_created_date (static, known before any decision window) — no leakage.
- low_ctr_flag uses only February's gsc_clicks and gsc_impressions (prior month, closed before March) — no leakage.
- No column derived from the March decline label (clicks_mar, clicks_diff, or declined itself) was used anywhere in the score, reason_code, or action. The label was only used earlier, separately, to verify Signals 1 and 2 — never as a scoring input.
- No product/system flags (e.g. optimization_eligible_date, is_deleted) were used — consistent with these being excluded in last week's data contract for the same leakage reason.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.